# BVI-Net -- Skin / Liver / Brain, one notebook

Runs all three organs from the base paper on a Kaggle GPU: ISIC2018 (skin),
LiTS17 (liver), BraTS19 (brain). No dataset is downloaded inside this
notebook -- everything comes from Kaggle's "+ Add Input" mount, which is
already local to the kernel with no download step.

**Each organ's section is independent** -- run only the section(s) you have
the raw dataset attached for. Skip a section entirely if you don't need to
(re)train that organ.

**The whole point of saving checkpoints at the end of each section**: once
`checkpoints/bvi_net_<organ>_best.pt` exists, the frontend backend
(`backend/app.py`) loads it directly for inference -- no retraining needed
on every request, or ever again unless you want to improve the model.

## Datasets to attach via "+ Add Input" (do this before running a section)

1. **Skin**: `tschandl/isic2018-challenge-task1-data-segmentation` (or
   whichever ISIC2018 Task 1 mirror you used previously).
2. **Liver**: `javariatahir123/lits17-liver-tumor-segmentation`.
3. **Brain**: whichever BraTS19 mirror you have licensed access to (official
   CBICA/Synapse export re-uploaded as a private Kaggle dataset, or an
   equivalently-licensed public Kaggle mirror if one exists).

After attaching, run `!ls /kaggle/input` and update `SKIN_RAW_DIR` /
`LIVER_RAW_DIR` / `BRATS_RAW_DIR` in the setup cell below to match the
actual mounted folder name -- Kaggle slugs don't always mount under the
exact slug string.

## 0. Setup (run once, shared by all three organs)

In [ ]:
!pip install causal-conv1d mamba-ssm --no-build-isolation

In [ ]:
!git clone https://github.com/Ghanasree-S/BVI-Net-ISIC.git
%cd BVI-Net-ISIC
!pip install -q -r requirements.txt

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)

# Adjust these to match whatever Kaggle actually mounts your attached
# datasets as -- run `!ls /kaggle/input` after attaching each dataset to
# confirm the exact folder name before running that organ's section.
SKIN_RAW_DIR = "/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation"
LIVER_RAW_DIR = "/kaggle/input/lits17-liver-tumor-segmentation"
BRATS_RAW_DIR = "/kaggle/input/brats19-dataset"  # placeholder -- update once you have a source

## 1. Skin -- ISIC2018

27,312 params, Dice 0.8631 previously. Uses the training-pool self-split
(no official-validation-set download) so this section only needs the
Kaggle input attached -- nothing pulled from the internet.

In [ ]:
!python data/prepare_isic_a.py \
  --raw_dir {SKIN_RAW_DIR} \
  --out_dir data/isic2018a \
  --fraction 1.0

In [ ]:
!python train.py --dataset isic --data_dir data/isic2018a \
  --epochs 50 --batch_size 8 --lr 0.001 --patience 15 \
  --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
!python evaluate.py --dataset isic --checkpoint checkpoints/best.pt --data_dir data/isic2018a \
  --visualize --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
import shutil
shutil.copy("checkpoints/best.pt", "checkpoints/bvi_net_skin_best.pt")
print("Saved checkpoints/bvi_net_skin_best.pt")

## 2. Liver -- LiTS17

Requires the LiTS17 Kaggle dataset attached (see setup cell above). Two
output channels (Liver, Tumor), 1-channel grayscale CT input, 448x448.

In [ ]:
!python data/prepare_lits.py \
  --raw_dir {LIVER_RAW_DIR} \
  --out_dir data/lits17a

In [ ]:
!python train.py --dataset lits --data_dir data/lits17a \
  --epochs 50 --batch_size 8 --lr 0.001 --patience 15 \
  --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
!python evaluate.py --dataset lits --checkpoint checkpoints/best.pt --data_dir data/lits17a \
  --visualize --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
import shutil
shutil.copy("checkpoints/best.pt", "checkpoints/bvi_net_liver_best.pt")
print("Saved checkpoints/bvi_net_liver_best.pt")

## 3. Brain -- BraTS19 (LGG only)

Requires an approved BraTS19 source attached (see setup cell above -- data
use agreement, not automatable). Three nested output channels (WT, TC,
ET), 4-channel MRI input (T1, T1ce, T2, FLAIR stacked), 240x240.

In [ ]:
!python data/prepare_brats.py \
  --raw_dir {BRATS_RAW_DIR} \
  --out_dir data/brats19a

In [ ]:
!python train.py --dataset brats --data_dir data/brats19a \
  --epochs 50 --batch_size 8 --lr 0.001 --patience 15 \
  --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
!python evaluate.py --dataset brats --checkpoint checkpoints/best.pt --data_dir data/brats19a \
  --visualize --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
import shutil
shutil.copy("checkpoints/best.pt", "checkpoints/bvi_net_brain_best.pt")
print("Saved checkpoints/bvi_net_brain_best.pt")

## 4. Package all trained checkpoints for download

Run this after whichever sections above you completed. Zips only the
organ-specific checkpoints that actually exist yet -- download the zip
from Kaggle's Output panel, then drop the .pt files into your local
`checkpoints/` folder so `backend/app.py` can serve real inference for
each organ without ever retraining again.

In [ ]:
import zipfile
from pathlib import Path

ckpt_dir = Path("checkpoints")
saved = sorted(p.name for p in ckpt_dir.glob("bvi_net_*_best.pt"))
print("Checkpoints ready to download:", saved or "(none yet -- run at least one organ section above)")

if saved:
    with zipfile.ZipFile("/kaggle/working/bvi_net_checkpoints.zip", "w") as zf:
        for name in saved:
            zf.write(ckpt_dir / name, arcname=name)
    print("Wrote /kaggle/working/bvi_net_checkpoints.zip")